In [1]:
import h5py
import numpy as np
import torch
from pathlib import Path
import os
import json

def convert_trajectories_to_hdf5(
    trajectories_dir: str,
    output_path: str,
    normalize_actions: bool = False,
    min_trajectory_length: int = 230
):
    """
    Convert imitation learning trajectories (.pt files) to HDF5 format for robomimic.
    Converts 4-channel stacked frames to 3-channel RGB by taking the most recent frame.
    """
    
    # Find all .pt files
    demo_files = sorted(Path(trajectories_dir).glob("demos*.pt"))
    
    if not demo_files:
        raise ValueError(f"No .pt files found in {trajectories_dir}")
    
    print(f"Found {len(demo_files)} files")
    
    # Load all trajectories
    all_trajectories = []
    filtered_trajectories = []
    
    for file_path in demo_files:
        try:
            trajs = torch.load(file_path, weights_only=False)
            original_count = len(trajs)
            filtered_count = 0
            
            for traj in trajs:
                traj_length = len(traj.acts)
                if traj_length >= min_trajectory_length:
                    all_trajectories.append(traj)
                else:
                    filtered_count += 1
            
            if filtered_count > 0:
                filtered_trajectories.append((file_path.name, filtered_count, original_count))
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            continue
    
    print(f"Total trajectories loaded: {len(all_trajectories)}")
    
    if filtered_trajectories:
        print(f"\nFiltered trajectories (shorter than {min_trajectory_length} steps):")
        for filename, filtered_count, original_count in filtered_trajectories:
            print(f"  {filename}: {filtered_count}/{original_count} trajectories filtered")
    
    if len(all_trajectories) == 0:
        raise ValueError(f"No trajectories with length >= {min_trajectory_length} found!")
    
    # Create output directory
    output_dir = Path(output_path).parent
    os.makedirs(output_dir, exist_ok=True)
    
    # Create HDF5 file
    with h5py.File(output_path, 'w') as f:
        # Main group
        data_grp = f.create_group("data")
        
        total_samples = sum(len(traj.acts) for traj in all_trajectories)
        data_grp.attrs["total"] = total_samples
        print(f"\nTotal samples: {total_samples}")
        
        # Add environment metadata
        env_args = {
            "env_name": "ResidentEvil",
            "type": "gym",
            "env_kwargs": {
                "observation_space": "image_3_128x128",
                "action_space": "multibinary_18"
            }
        }
        data_grp.attrs["env_args"] = json.dumps(env_args)
        data_grp.attrs["env_name"] = "ResidentEvil"
        data_grp.attrs["type"] = "gym"
        
        kept_trajectories = 0
        
        # Process each trajectory
        for demo_idx, traj in enumerate(all_trajectories):
            demo_name = f"demo_{demo_idx}"
            demo_grp = data_grp.create_group(demo_name)
            
            # Extract data
            obs = np.array(traj.obs)
            acts = np.array(traj.acts)
            traj_length = len(acts)
            
            print(f"\nProcessing {demo_name}...")
            
            # ===== Fix actions shape =====
            print(f"  Actions shape before: {acts.shape}")
            
            # Remove singleton dimensions
            if acts.ndim == 3:
                if acts.shape[1] == 1:
                    acts = acts.squeeze(1)
                elif acts.shape[2] == 1:
                    acts = acts.squeeze(-1)
            
            if acts.ndim == 1:
                acts = acts.reshape(-1, 1)
            
            if acts.ndim != 2:
                raise ValueError(f"Actions shape {acts.shape} is not 2D")
            
            print(f"  Actions shape after: {acts.shape}")
            
            # Store number of samples
            demo_grp.attrs["num_samples"] = traj_length
            
            # Create dones array
            dones = np.zeros(traj_length, dtype=bool)
            dones[-1] = traj.terminal if hasattr(traj, 'terminal') else True
            
            # ===== Fix observations shape =====
            print(f"  Observations shape before: {obs.shape}")
            
            # Remove singleton dimensions
            if obs.ndim == 5 and obs.shape[1] == 1:
                obs = obs.squeeze(1)
            
            # Handle (T, C, H, W) format
            if obs.ndim == 4 and obs.shape[1] in [1, 3, 4]:
                # Convert to (T, H, W, C)
                obs = obs.transpose(0, 2, 3, 1)
                print(f"  After transpose: {obs.shape}")
            
            # Now obs should be (T, H, W, C) with C = 4 (stacked frames)
            # Convert to RGB by taking the last 3 channels (most recent frame)
            if obs.shape[-1] == 4:
                # Option: Take the last 3 channels as RGB
                # If your 4 channels are 4 grayscale frames stacked, we take the last 3
                # to get a 3-channel RGB-like image
                obs = obs[..., 1:4]  # Take channels 1,2,3 (skip the oldest)
                print(f"  After converting 4 channels to 3: {obs.shape}")
            elif obs.shape[-1] == 1:
                # Grayscale single channel - repeat to make 3 channels
                obs = np.repeat(obs, 3, axis=-1)
                print(f"  Converted grayscale to RGB: {obs.shape}")
            
            # Final check: must have 3 channels
            if obs.shape[-1] != 3:
                raise ValueError(f"Observations have {obs.shape[-1]} channels, expected 3. Shape: {obs.shape}")
            
            # Ensure uint8
            if obs.dtype != np.uint8:
                obs = obs.astype(np.uint8)
            
            print(f"  Final observations shape: {obs.shape}")
            print(f"  Observation range: [{obs.min()}, {obs.max()}]")
            
            # Process actions
            if normalize_actions:
                actions = acts.astype(np.float32) * 2 - 1
            else:
                actions = acts.astype(np.float32)
            
            # Save datasets
            demo_grp.create_dataset(
                "actions", 
                data=actions,
                chunks=True,
                compression="gzip"
            )
            
            rewards = np.zeros(traj_length, dtype=np.float32)
            demo_grp.create_dataset(
                "rewards", 
                data=rewards,
                chunks=True,
                compression="gzip"
            )
            
            demo_grp.create_dataset(
                "dones", 
                data=dones,
                chunks=True,
                compression="gzip"
            )
            
            obs_grp = demo_grp.create_group("obs")
            obs_grp.create_dataset(
                "image",
                data=obs,
                chunks=(1, *obs.shape[1:]),
                compression="gzip"
            )
            
            kept_trajectories += 1
            if kept_trajectories % 20 == 0:
                print(f"  Processed {kept_trajectories}/{len(all_trajectories)} trajectories...")
        
        data_grp.attrs["num_demos"] = kept_trajectories
        data_grp.attrs["num_samples"] = total_samples
    
    print(f"\n✅ Conversion completed!")
    print(f"   File saved to: {output_path}")
    print(f"   Kept trajectories: {kept_trajectories}")
    print(f"   Total samples: {total_samples}")
    return output_path



# Remove old file if exists
if os.path.exists("datasets/resident_evil_dataset.hdf5"):
    os.remove("datasets/resident_evil_dataset.hdf5")
    print("Removed old file")

# Create datasets directory
os.makedirs("datasets", exist_ok=True)

# Convert
convert_trajectories_to_hdf5(
    trajectories_dir="demos/",
    output_path="datasets/resident_evil_dataset.hdf5",
    normalize_actions=False,  # Keep actions as 0/1
    min_trajectory_length=230
)

Removed old file
Found 18 files
Total trajectories loaded: 174

Filtered trajectories (shorter than 230 steps):
  demos1.pt: 2/10 trajectories filtered
  demos14.pt: 1/10 trajectories filtered
  demos5.pt: 1/10 trajectories filtered
  demos7.pt: 2/10 trajectories filtered

Total samples: 192416

Processing demo_0...
  Actions shape before: (1216, 1, 18)
  Actions shape after: (1216, 18)
  Observations shape before: (1217, 1, 128, 128, 4)
  After converting 4 channels to 3: (1217, 128, 128, 3)
  Final observations shape: (1217, 128, 128, 3)
  Observation range: [11, 253]

Processing demo_1...
  Actions shape before: (998, 1, 18)
  Actions shape after: (998, 18)
  Observations shape before: (999, 1, 128, 128, 4)
  After converting 4 channels to 3: (999, 128, 128, 3)
  Final observations shape: (999, 128, 128, 3)
  Observation range: [11, 254]

Processing demo_2...
  Actions shape before: (885, 1, 18)
  Actions shape after: (885, 18)
  Observations shape before: (886, 1, 128, 128, 4)
  Af

'datasets/resident_evil_dataset.hdf5'